# AI Resume Screener – Model Analysis Notebook

This notebook walks through the full pipeline: data generation, preprocessing, model training, and evaluation.


In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import re, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print('✓ Libraries loaded')

## 1. Generate & Load Dataset

In [ ]:
os.chdir('../data')
from generate_dataset import generate_dataset
df = generate_dataset(500)
os.chdir('../notebooks')
df.head(3)

## 2. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = df['category'].value_counts()
colors = ['#6366f1','#8b5cf6','#a78bfa','#c4b5fd','#ddd6fe']
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Resume Count per Category', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Text Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s\+\#]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['resume'].apply(clean_text)
print(f'Avg word count: {df["clean"].apply(lambda x: len(x.split())).mean():.0f} words/resume')
df[['category','clean']].head(2)

## 4. TF-IDF Vectorization

In [ ]:
X = df['clean']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=15000, sublinear_tf=True, min_df=2)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'Train shape: {X_train_vec.shape} | Test shape: {X_test_vec.shape}')

## 5. Model Training & Evaluation

In [ ]:
models = {
    'SVM (Linear)': SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42, multi_class='multinomial'),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    preds = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    results[name] = {'clf': clf, 'preds': preds, 'acc': acc}
    print(f'{name}: {acc*100:.2f}%')

best_name = max(results, key=lambda n: results[n]['acc'])
print(f'\n✓ Best: {best_name} ({results[best_name]["acc"]*100:.2f}%)')

## 6. Confusion Matrix

In [ ]:
best = results[best_name]
cm = confusion_matrix(y_test, best['preds'], labels=list(models.keys())[:0] + sorted(y.unique()))
cm = confusion_matrix(y_test, best['preds'])
labels = sorted(y.unique())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=labels, yticklabels=labels, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title(f'Confusion Matrix – {best_name}', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, best['preds']))

## 7. Cross-Validation

In [ ]:
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=15000, sublinear_tf=True, min_df=2)),
    ('clf', SVC(kernel='linear', C=1.0, probability=True, random_state=42)),
])
cv = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print(f'Folds: {[f"{s*100:.1f}%" for s in cv]}')

## 8. Top TF-IDF Features per Category

In [ ]:
lr = results['Logistic Regression']['clf']
feature_names = np.array(tfidf.get_feature_names_out())
categories = sorted(y.unique())

fig, axes = plt.subplots(1, len(categories), figsize=(18, 5))
for ax, cat, coef in zip(axes, categories, lr.coef_):
    top_idx = np.argsort(coef)[-10:][::-1]
    top_words = feature_names[top_idx]
    top_vals  = coef[top_idx]
    ax.barh(top_words[::-1], top_vals[::-1], color='#6366f1', alpha=0.8)
    ax.set_title(cat, fontweight='bold', fontsize=10)
    ax.set_xlabel('Coefficient')
plt.suptitle('Top 10 TF-IDF Features per Category', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()